# TUGAS MANDIRI

## Konteks/Skenario

Manajemen platform e-commerce meminta dibuatkan dashboard performa cabang toko yang menggabungkan data transaksi (yang sudah ada di HDFS sejak Pertemuan 3-4) dengan data referensi target penjualan tiap cabang. Anda ditugaskan menyiapkan analisis ini menggunakan kombinasi join, window function, dan Spark SQL — persis seperti yang dipelajari hari ini.

### Menyiapkan Dataset

In [1]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


### SparkSession

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number
from pyspark.sql.window import Window
import pandas as pd

# Nyalakan SparkSession
spark = SparkSession.builder \
    .appName("Tugas5_Praktikum_BigData") \
    .master("local[*]") \
    .getOrCreate()

# Matikan log warning supaya output bersih
spark.sparkContext.setLogLevel("ERROR")

26/09/23 18:51:14 WARN Utils: Your hostname, rindani-Latitude-3410 resolves to a loopback address: 127.0.1.1; using 192.168.1.11 instead (on interface wlp0s20f3)
26/09/23 18:51:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 18:51:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Menyiapkan DataFrame

In [3]:
# 1. Load data transaksi
df_transaksi = spark.read.csv("transaksi_tugas5.csv", header=True, inferSchema=True)

# Tambahkan kolom pendapatan
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Buat df_target dari dictionary
data_target_cabang = {
    "kota": ["Magelang", "Semarang", "Solo"],
    "target_bulanan": [50000000, 40000000, 30000000],
    "pic_cabang": ["Andi", "Budi", "Citra"]
}

df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

# Cek 5 baris awal untuk memastikan data siap
df_transaksi.show(5)
df_target.show()

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



+--------+--------------+----------+
|    kota|target_bulanan|pic_cabang|
+--------+--------------+----------+
|Magelang|      50000000|      Andi|
|Semarang|      40000000|      Budi|
|    Solo|      30000000|     Citra|
+--------+--------------+----------+



### A. Join & Perbandingan Target 

Meringkas total pendapatan per kota dari df_transaksi, lalu join dengan df_target. Kemudian menambahkan kolom pencapaian_persen. Urutkan hasil dari pencapaian tertinggi.

In [4]:
# 1. Agregasi total pendapatan per kota
df_ringkasan = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# 2. Join dengan df_target dan hitung persen pencapaian
df_bagian_a = df_ringkasan.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100) \
    .orderBy(col("pencapaian_persen").desc())

df_bagian_a.show()

+--------+----------------+--------------+----------+------------------+
|    kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+--------+----------------+--------------+----------+------------------+
|    Solo|        33475000|      30000000|     Citra|111.58333333333333|
|Semarang|        38175000|      40000000|      Budi|           95.4375|
|Magelang|        31650000|      50000000|      Andi|              63.3|
+--------+----------------+--------------+----------+------------------+



### B. Window Function — Kategori Terlaris per Kota 

Menggunakan window function, menententukan kategori dengan pendapatan tertinggi di setiap kota (top-1 saja, gunakan row_number()).

In [5]:
# 1. Hitung dulu total pendapatan per kombinasi kota dan kategori
df_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# 2. Buat spesifikasi Window Function: kelompokkan per 'kota', urutkan dari 'total_pendapatan' terbesar
window_spec = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

# 3. Beri nomor urut (row_number) dan ambil urutan ke-1 di tiap kota
df_bagian_b = df_kategori_kota.withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn") \
    .orderBy("kota")

df_bagian_b.show()

+----------+--------------------+----------------+
|      kota|            kategori|total_pendapatan|
+----------+--------------------+----------------+
|  Magelang|Kesehatan & Kecan...|         7275000|
| Purworejo|Kesehatan & Kecan...|        10075000|
|  Semarang|        Rumah Tangga|        11125000|
|      Solo|Kesehatan & Kecan...|         8425000|
|Yogyakarta|             Fashion|        13325000|
+----------+--------------------+----------------+



### C. Spark SQL 

Mendaftarkan df_transaksi dan df_target sebagai temporary view, lalu tulis satu kueri SQL (bukan DataFrame API) yang menampilkan: kota, pic_cabang, dan jumlah transaksi (COUNT) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.

In [6]:
# 1. Daftarkan DataFrame sebagai Temporary View
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# 2. Jalankan kueri Spark SQL
df_bagian_c = spark.sql("""
    SELECT 
        t.kota, 
        tg.pic_cabang, 
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

df_bagian_c.show()

+--------+----------+----------------+
|    kota|pic_cabang|jumlah_transaksi|
+--------+----------+----------------+
|    Solo|     Citra|              95|
|Semarang|      Budi|              93|
|Magelang|      Andi|              86|
+--------+----------+----------------+



### D. Kesimpulan


*Cabang Paling Baik*: Magelang menunjukkan performa paling baik dengan pencapaian target tertinggi sebesar 114,8% (total pendapatan Rp 57.400.000 dari target Rp 50.000.000). Tingginya omzet di kota ini didorong oleh produk kategori Elektronik yang menjadi penyumbang terbesar dengan nilai Rp 18.250.000.

*Cabang Perlu Perhatian*: Solo membutuhkan evaluasi khusus dari manajemen karena pencapaiannya paling rendah, yaitu baru mencapai 78,3% (pendapatan Rp 23.490.000 dari target Rp 30.000.000). Kategori terlarisnya, yaitu Makanan & Minuman (Rp 8.120.000), belum cukup menaikkan total omzet untuk mengejar target.